# Object Detection using Federated Learning

This notebook implements Federated Learning for object detection using YOLOv8. It splits the COCO128 dataset among multiple clients, performs local training, and uses Federated Averaging (FedAvg) to aggregate the model weights globally.


## 1. Environment Setup
Install the required dependencies, primarily `ultralytics` for YOLOv8.


In [ ]:
!pip install ultralytics torch torchvision opencv-python matplotlib pyyaml


## 2. Dataset Preparation
Download the COCO128 dataset. We will use this as our base dataset to distribute among clients.


In [ ]:
from ultralytics.utils.downloads import download
from pathlib import Path
import os
import shutil
import yaml

# Download COCO128 dataset
url = "https://ultralytics.com/assets/coco128.zip"
download(url, dir=".")


## 3. Client Data Splitting
Distribute the dataset among multiple clients simulating a federated environment.


In [ ]:
import random
import copy
import glob
from PIL import Image

NUM_CLIENTS = 2

def split_dataset_into_clients(images_source, labels_source, num_clients=2):
    # Get all image paths
    image_paths = glob.glob(os.path.join(images_source, '*.jpg'))
    random.shuffle(image_paths)
    
    # Calculate split size
    split_size = len(image_paths) // num_clients
    
    clients_data = []
    
    for i in range(num_clients):
        client_dir = f"client_{i}_data"
        client_images_dir = os.path.join(client_dir, "images", "train")
        client_labels_dir = os.path.join(client_dir, "labels", "train")
        
        os.makedirs(client_images_dir, exist_ok=True)
        os.makedirs(client_labels_dir, exist_ok=True)
        
        # Allocate images to client
        start_idx = i * split_size
        end_idx = (i + 1) * split_size if i != num_clients - 1 else len(image_paths)
        client_image_subset = image_paths[start_idx:end_idx]
        
        for img_path in client_image_subset:
            # Copy image
            shutil.copy(img_path, client_images_dir)
            
            # Copy label
            img_filename = os.path.basename(img_path)
            label_filename = img_filename.replace('.jpg', '.txt')
            label_path = os.path.join(labels_source, label_filename)
            
            if os.path.exists(label_path):
                shutil.copy(label_path, client_labels_dir)
                
        # Create YAML config for the client
        yaml_config = {
            'path': os.path.abspath(client_dir),
            'train': 'images/train',
            'val': 'images/train', # Use train as val for simplicity in this demo
            'names': {i: str(i) for i in range(80)} # Placeholder for COCO classes
        }
        
        # Read the original coco128.yaml to get class names
        try:
            with open('coco128/coco128.yaml', 'r') as f:
                orig_yaml = yaml.safe_load(f)
                yaml_config['names'] = orig_yaml.get('names', yaml_config['names'])
        except Exception as e:
            pass
            
        yaml_path = f"client_{i}_dataset.yaml"
        with open(yaml_path, 'w') as f:
            yaml.dump(yaml_config, f)
            
        clients_data.append(yaml_path)
        print(f"Client {i} created with {len(client_image_subset)} images.")
        
    return clients_data

clients_yaml = split_dataset_into_clients('coco128/images/train2017', 'coco128/labels/train2017', NUM_CLIENTS)



## 4. Federated Averaging Algorithm
Implement FedAvg to aggregate model weights from all participating clients.


In [ ]:
import torch

def federated_averaging(client_weights_list):
    """
    Averages the weights from multiple clients to create the updated global model weights.
    """
    avg_weights = copy.deepcopy(client_weights_list[0])
    
    for key in avg_weights.keys():
        for i in range(1, len(client_weights_list)):
            avg_weights[key] += client_weights_list[i][key]
            
        # Divide by number of clients
        if avg_weights[key].is_floating_point():
            avg_weights[key] = avg_weights[key] / len(client_weights_list)
        else:
            # Handle integer tensors like num_batches_tracked
            avg_weights[key] = avg_weights[key] // len(client_weights_list)
            
    return avg_weights


## 5. Global Model Initialization & Training Loop
Initialize the global YOLOv8 model and perform the federated training process over multiple rounds.


In [ ]:
from ultralytics import YOLO

# Initialize Global Model
global_model = YOLO('yolov8n.pt')

NUM_ROUNDS = 20
LOCAL_EPOCHS = 10
IMGSZ = 320

print("Starting Federated Training Process...")

for round_num in range(NUM_ROUNDS):
    print(f"\n{'='*30}\nRound {round_num + 1}/{NUM_ROUNDS}\n{'='*30}")
    
    client_weights_list = []
    
    # Pass global weights directly via state_dict to avoid save/load issues
    global_state_dict = copy.deepcopy(global_model.model.state_dict())
    
    for client_id in range(NUM_CLIENTS):
        print(f"\n--- Training Client {client_id} ---")
        yaml_config = clients_yaml[client_id]
        
        # Load the global model for the client by applying the state_dict
        client_model = YOLO('yolov8n.pt')
        client_model.model.load_state_dict(global_state_dict)
        
        # Train locally
        results = client_model.train(
            data=yaml_config,
            epochs=LOCAL_EPOCHS,
            imgsz=IMGSZ,
            device='cpu', # CPU is specified in the report's limitations/performance summary
            project=f"runs_client_{client_id}",
            name=f"round_{round_num}",
            exist_ok=True,
            verbose=False,
            plots=False,  # Prevent generating confusion matrices and curves
            save=False,   # Prevent saving unnecessary models/metrics
            val=False     # Disable validation to speed up training
        )
        
        # Extract the trained weights
        client_weights = copy.deepcopy(client_model.model.state_dict())
        client_weights_list.append(client_weights)
        
    # Aggregate weights using FedAvg
    print("\nAggregating weights from clients...")
    aggregated_weights = federated_averaging(client_weights_list)
    
    # Update global model
    global_model.model.load_state_dict(aggregated_weights)
    
print("\nFederated Training Completed!")


## 6. Results and Evaluation
Evaluate the trained global model on test images and display the bounding box predictions.


In [ ]:
%matplotlib inline
import matplotlib.pyplot as plt
import cv2

# Select some test images from the original dataset
test_images = glob.glob('coco128/images/train2017/*.jpg')[:4] # Evaluate on 4 images as per report

print(f"Evaluating model on {len(test_images)} test images...")

# Make predictions silently and enforce inline plotting
results = global_model.predict(
    source=test_images, 
    conf=0.40, # Confidence threshold from the report
    save=True,
    verbose=False # Disable the text output for each prediction
)

import os

# Plotting the tested images with bounding boxes as requested
for i, (result, img_path) in enumerate(zip(results, test_images)):
    filename = os.path.basename(img_path)
    num_objects = len(result.boxes)
    
    # Original image
    orig_img = cv2.imread(img_path)
    orig_img_rgb = cv2.cvtColor(orig_img, cv2.COLOR_BGR2RGB)
    
    # Annotated image
    annotated_img = result.plot()
    annotated_img_rgb = cv2.cvtColor(annotated_img, cv2.COLOR_BGR2RGB)
    
    print(f"Detailed View {i+1}: {filename}")
    print("========================================")
    
    fig, axes = plt.subplots(1, 2, figsize=(12, 6))
    
    axes[0].imshow(orig_img_rgb)
    axes[0].set_title("Original Image", fontweight='bold', fontsize=10)
    axes[0].axis('off')
    
    axes[1].imshow(annotated_img_rgb)
    axes[1].set_title(f"Predictions: {num_objects} Objects Detected", fontweight='bold', color='green', fontsize=10)
    axes[1].axis('off')
    
    plt.show()
    print("\n")

# Performance summary
total_detected = sum(len(r.boxes) for r in results)
print("DETECTION STATISTICS")
print("----------------------------------------")
print(f"Total Images Tested: {len(test_images)}")
print(f"Total Objects Detected: {total_detected}")
print(f"Average Objects per Image: {total_detected / len(test_images):.2f}")

# Plotting the Detection Statistics Graph (Bar Chart)
image_names = [os.path.basename(img) for img in test_images]
object_counts = [len(r.boxes) for r in results]

plt.figure(figsize=(10, 5))
bars = plt.bar(image_names, object_counts, color='#8cb4d6', edgecolor='gray')

# Add text on top of bars
for bar in bars:
    yval = bar.get_height()
    plt.text(bar.get_x() + bar.get_width()/2, yval + 0.05, int(yval), ha='center', va='bottom', fontsize=9, fontweight='bold')

plt.title("Detection Count per Image", fontweight='bold', fontsize=12)
plt.ylabel("Number of Objects Detected", fontsize=10)
plt.xlabel("Image", fontsize=10)
plt.xticks(rotation=45, ha='right', fontsize=8)
plt.grid(axis='y', linestyle='-', alpha=0.3)

# Remove top and right borders
plt.gca().spines['top'].set_visible(False)
plt.gca().spines['right'].set_visible(False)

plt.tight_layout()
plt.show()
